# MAGAIL-C — Full Analytics NotebookExpert corpus (52 episodes) vs 4 ablation conditions (NC/C/SC/C+KL, 4 representativeseeds x 60 episodes each = 960 agent episodes), plus the 8-seed x 500-episodesignificance-test data. One chart per cell, organized to match the visualizationcatalogue's sections A–I.**Before running:** update `LAUNCH_ID` and the two directory paths below to matchyour actual run.

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport glob, os, jsonfrom scipy.spatial import ConvexHull, Voronoi, voronoi_plot_2dfrom scipy.spatial.distance import pdist%matplotlib inlineplt.rcParams['figure.dpi'] = 100np.random.seed(0)

In [ ]:
LAUNCH_ID = "matrix_0911ecd0"EXPERT_DIR = os.path.expanduser("~/dissertation/context-aware-magail-grf/data/demonstrations/episodes")AGENT_DIR = f"results_v2/stage4_rollouts/{LAUNCH_ID}/episodes"STATS_DIR = f"results_v2/stage3_full_matrix/{LAUNCH_ID}"CONDITIONS = ["NC", "C", "SC", "C+KL"]COND_COLORS = {"NC": "tab:orange", "C": "tab:green", "SC": "tab:red",              "C+KL": "tab:blue", "expert": "black", "baseline": "gray"}OUTFIELD = [1, 2, 3, 4]  # index 0 = goalkeeper

## Data loadingOne .npz per episode, loaded once into memory. Position/spatialcharts iterate episode-by-episode (memory-safe at any scale); the scalardataframe below is vectorized per-episode (NOT per-step) for speed.

In [ ]:
def load_episode_files(pattern):    files = sorted(glob.glob(pattern))    return {os.path.basename(f): dict(np.load(f, allow_pickle=True)) for f in files}print("Loading expert episodes...")expert_episodes = load_episode_files(os.path.join(EXPERT_DIR, "*.npz"))print(f"  {len(expert_episodes)} expert episodes")agent_episodes = {}for cond in CONDITIONS:    pattern = os.path.join(AGENT_DIR, f"{cond}_seed*_ep*.npz")    agent_episodes[cond] = load_episode_files(pattern)    print(f"  {cond}: {len(agent_episodes[cond])} episodes")

In [ ]:
def classify_time_zone_arr(elapsed_min):    return np.select([elapsed_min < 20, elapsed_min < 70], ['early', 'mid'], default='late')def build_scalar_df(episodes_dict, source_label, condition_label=None):    """Vectorized PER EPISODE (not per step) -- fast even at ~3M total steps.    context_region is NEVER read from the stored .npz field -- always    recomputed here, correctly, from steps_left/score directly."""    dfs = []    for name, d in episodes_dict.items():        n = len(d['steps_left'])        t_norm = d['steps_left'] / 3001.0        elapsed_min = (1 - t_norm) * 90.0        dscore = d['score_left'] - d['score_right']        sprint = d['sticky_actions'][:, 8].astype(int)        seed = int(d['seed']) if 'seed' in d else -1        time_zone = classify_time_zone_arr(elapsed_min)        score_zone = np.select([dscore > 0, dscore < 0], ['win', 'loss'], default='draw')        dfs.append(pd.DataFrame({            'source': source_label, 'condition': condition_label or source_label,            'file': name, 'seed': seed, 'step': np.arange(n),            'elapsed_min': elapsed_min, 'time_zone': time_zone,            'dscore': dscore, 'score_zone': score_zone,            'sprint': sprint, 'has_possession': d['has_possession'].astype(bool),        }))    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()print("Building scalar dataframes...")expert_df = build_scalar_df(expert_episodes, 'expert')agent_dfs = {cond: build_scalar_df(agent_episodes[cond], 'agent', cond) for cond in CONDITIONS}print(f"expert: {len(expert_df):,} rows")for cond in CONDITIONS:    print(f"{cond}: {len(agent_dfs[cond]):,} rows")

In [ ]:
# Stats JSONs from the ablation matrix -- small, load directlywith open(os.path.join(STATS_DIR, 'all_runs_raw.json')) as f:    all_runs = json.load(f)with open(os.path.join(STATS_DIR, 'condition_summary.json')) as f:    cond_summary = json.load(f)cond_summary_dict = {row['condition']: row for row in cond_summary}sig_path = os.path.join(STATS_DIR, 'significance_tests.json')sig_tests = json.load(open(sig_path)) if os.path.exists(sig_path) else []runs_df = pd.DataFrame(all_runs)runs_df = runs_df[runs_df['condition_label'].isin(CONDITIONS)]print(f"{len(runs_df)} rows in runs_df (should be 32 = 4 conditions x 8 seeds)")

---# A. SPATIAL / TACTICAL

### A1 — Position density heatmap, expert + all 4 conditions

In [ ]:
def accumulate_position_heatmap(episodes_dict, players=OUTFIELD, grid_x=40, grid_y=24):    x_edges = np.linspace(-1, 1, grid_x + 1); y_edges = np.linspace(-0.42, 0.42, grid_y + 1)    heat = np.zeros((grid_y, grid_x))    for name, d in episodes_dict.items():        pos = d['left_team'][:, players, :]        h, _, _ = np.histogram2d(pos[:,:,0].ravel(), pos[:,:,1].ravel(), bins=[x_edges, y_edges])        heat += h.T    return heat, x_edges, y_edgesfig, axes = plt.subplots(1, 5, figsize=(22, 5))sources = [('expert', expert_episodes)] + [(c, agent_episodes[c]) for c in CONDITIONS]for ax, (label, eps) in zip(axes, sources):    heat, xe, ye = accumulate_position_heatmap(eps)    ax.imshow(heat, origin='lower', extent=[-1,1,-0.42,0.42], cmap='inferno', aspect='auto')    ax.set_title(f'{label}\nposition density'); ax.set_xlim(-1,1); ax.set_ylim(-0.42,0.42)plt.tight_layout(); plt.show()

### A3 — Difference heatmap (agent minus expert), LOCKED-BUDGET FIGURESigned, diverging colourmap: where does agent behaviour over/under-representrelative to human play, position-by-position on the pitch.

In [ ]:
def normalized_heatmap(episodes_dict, players=OUTFIELD, grid_x=40, grid_y=24):    heat, xe, ye = accumulate_position_heatmap(episodes_dict, players, grid_x, grid_y)    return heat / heat.sum(), xe, yeexpert_heat, xe, ye = normalized_heatmap(expert_episodes)fig, axes = plt.subplots(1, 4, figsize=(20, 5))for ax, cond in zip(axes, CONDITIONS):    agent_heat, _, _ = normalized_heatmap(agent_episodes[cond])    diff = agent_heat - expert_heat    vmax = np.abs(diff).max()    im = ax.imshow(diff, origin='lower', extent=[-1,1,-0.42,0.42], cmap='RdBu_r',                   vmin=-vmax, vmax=vmax, aspect='auto')    ax.set_title(f'{cond} minus expert\n(red=agent-heavy, blue=agent-light)')    plt.colorbar(im, ax=ax, shrink=0.7)plt.tight_layout(); plt.show()

### A4 — Pitch control (Voronoi), one mid-match frame per condition

In [ ]:
def plot_voronoi_frame(d, t, ax, title=""):    left, right = d['left_team'][t], d['right_team'][t]    vor = Voronoi(np.vstack([left, right]))    voronoi_plot_2d(vor, ax=ax, show_vertices=False, line_colors='gray', line_width=0.5, point_size=0)    ax.scatter(left[:,0], left[:,1], c='tab:blue', s=80, zorder=3, label='controlled')    ax.scatter(right[:,0], right[:,1], c='tab:red', s=80, zorder=3, label='opponent')    ax.set_xlim(-1,1); ax.set_ylim(-0.42,0.42); ax.invert_yaxis(); ax.set_title(title); ax.legend(fontsize=8)fig, axes = plt.subplots(1, 4, figsize=(20, 5))for ax, cond in zip(axes, CONDITIONS):    name = list(agent_episodes[cond].keys())[0]    d = agent_episodes[cond][name]    plot_voronoi_frame(d, len(d['steps_left'])//2, ax, title=f'{cond} -- pitch control')plt.tight_layout(); plt.show()

### A6 — Formation shape overlay (convex hull, 5 snapshots per condition)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))for ax, cond in zip(axes, CONDITIONS):    name = list(agent_episodes[cond].keys())[0]    d = agent_episodes[cond][name]    for t in np.linspace(0, len(d['steps_left'])-1, 5).astype(int):        pos = d['left_team'][t, OUTFIELD]        hull = ConvexHull(pos)        for simplex in hull.simplices:            ax.plot(pos[simplex,0], pos[simplex,1], alpha=0.4)        ax.scatter(pos[:,0], pos[:,1], s=15, alpha=0.6)    ax.set_xlim(-1,1); ax.set_ylim(-0.42,0.42); ax.invert_yaxis()    ax.set_title(f'{cond} -- formation snapshots')plt.tight_layout(); plt.show()

### A7 — Convex hull area (MECHA) trajectory over match time

In [ ]:
def hull_area_series(d, players=OUTFIELD):    n = len(d['steps_left']); areas = np.full(n, np.nan)    for t in range(n):        try:            areas[t] = ConvexHull(d['left_team'][t, players]).volume        except Exception:            pass    return areasfig, ax = plt.subplots(figsize=(12, 5))for cond in CONDITIONS:    name = list(agent_episodes[cond].keys())[0]    d = agent_episodes[cond][name]    areas = hull_area_series(d)    elapsed = (1 - d['steps_left']/3001.0) * 90    ax.plot(elapsed, pd.Series(areas).rolling(50, min_periods=1).mean(),           label=cond, color=COND_COLORS[cond], alpha=0.85)ax.set_xlabel('elapsed minutes'); ax.set_ylabel('hull area (smoothed)')ax.legend(); ax.set_title('Formation compactness over match time (one representative episode/condition)')plt.tight_layout(); plt.show()

### A8 — Team centroid path (colour = time elapsed)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))for ax, cond in zip(axes, CONDITIONS):    name = list(agent_episodes[cond].keys())[0]    d = agent_episodes[cond][name]    centroid = d['left_team'][:, OUTFIELD].mean(axis=1)    ax.scatter(centroid[:,0], centroid[:,1], c=np.arange(len(centroid)), cmap='viridis', s=3)    ax.set_xlim(-1,1); ax.set_ylim(-0.42,0.42); ax.invert_yaxis()    ax.set_title(f'{cond} -- centroid path')plt.tight_layout(); plt.show()

### A9 — Centroid-to-ball distance over time (does shape track the ball or hold position?)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))for cond in CONDITIONS:    name = list(agent_episodes[cond].keys())[0]    d = agent_episodes[cond][name]    centroid = d['left_team'][:, OUTFIELD].mean(axis=1)    dist = np.linalg.norm(centroid - d['ball'][:, :2], axis=1)    elapsed = (1 - d['steps_left']/3001.0) * 90    ax.plot(elapsed, pd.Series(dist).rolling(50, min_periods=1).mean(), label=cond, color=COND_COLORS[cond])ax.set_xlabel('elapsed minutes'); ax.set_ylabel('centroid-ball distance (smoothed)')ax.legend(); ax.set_title('Centroid-vs-ball distance')plt.tight_layout(); plt.show()

### A10 — Zone occupation (defensive / middle / attacking third)

In [ ]:
def zone_occupation(episodes_dict, players=OUTFIELD):    counts = {'defensive': 0, 'middle': 0, 'attacking': 0}    for name, d in episodes_dict.items():        xs = d['left_team'][:, players, 0].ravel()        counts['defensive'] += int((xs < -1/3).sum())        counts['middle'] += int(((xs >= -1/3) & (xs <= 1/3)).sum())        counts['attacking'] += int((xs > 1/3).sum())    total = sum(counts.values())    return {k: 100*v/total for k, v in counts.items()}occ_data = {'expert': zone_occupation(expert_episodes)}for cond in CONDITIONS:    occ_data[cond] = zone_occupation(agent_episodes[cond])occ_df = pd.DataFrame(occ_data).Tocc_df[['defensive','middle','attacking']].plot(kind='bar', stacked=True, figsize=(10,5),                                                 color=['tab:red','tab:gray','tab:green'])plt.ylabel('% of player-steps'); plt.title('Zone occupation'); plt.xticks(rotation=0)plt.tight_layout(); plt.show()

---# B. NETWORK-BASED

### B1 — Inferred pass network, expert + all 4 conditions**Assumption flagged, verify before trusting:** for agent episodes, `action_captured`is `(T,4)`, and this assumes GRF indexes controlled players 1-4 (GK=0) to columns0-3 of that array in the same order. Confirm this against a known pass event beforeciting pass-count numbers from this cell.

In [ ]:
def inferred_pass_network(episodes_dict, is_agent, window=40):    pass_edges = {}    node_positions = {p: [] for p in OUTFIELD}    for name, d in episodes_dict.items():        n = len(d['steps_left'])        owned_team, owned_player = d['ball_owned_team'], d['ball_owned_player']        actions = d['action_captured']        for t in range(n):            if owned_team[t] != 0 or not (0 <= owned_player[t] < 5):                continue            op = int(owned_player[t])            if op in OUTFIELD:                node_positions[op].append(d['left_team'][t, op])            if is_agent:                if op == 0 or op > 4:                    continue                act = actions[t, op - 1]  # ASSUMPTION -- see markdown above            else:                if op not in OUTFIELD:                    continue                act = actions[t]            if act not in (9, 10, 11):                continue            passer = op            for look in range(t+1, min(t+window, n)):                if owned_team[look] == 1:                    break                if owned_team[look] == 0 and int(owned_player[look]) != passer and owned_player[look] in OUTFIELD:                    receiver = int(owned_player[look])                    pass_edges[(passer, receiver)] = pass_edges.get((passer, receiver), 0) + 1                    break    node_mean_pos = {p: (np.mean(node_positions[p], axis=0) if node_positions[p] else np.array([0,0])) for p in OUTFIELD}    return pass_edges, node_mean_posfig, axes = plt.subplots(1, 5, figsize=(24, 5))sources = [('expert', expert_episodes, False)] + [(c, agent_episodes[c], True) for c in CONDITIONS]for ax, (label, eps, is_agent) in zip(axes, sources):    edges, positions = inferred_pass_network(eps, is_agent)    max_c = max(edges.values()) if edges else 1    for (a,b), c in edges.items():        p1, p2 = positions[a], positions[b]        ax.plot([p1[0],p2[0]], [p1[1],p2[1]], color='gray', alpha=0.5, linewidth=1+3*c/max_c)    for p in OUTFIELD:        ax.scatter(*positions[p], s=200, zorder=3, edgecolor='black')        ax.annotate(str(p), positions[p], fontsize=9)    ax.set_xlim(-1,1); ax.set_ylim(-0.42,0.42); ax.invert_yaxis()    ax.set_title(f'{label}\n(n={sum(edges.values())} inferred passes)')plt.tight_layout(); plt.show()

---# C. RADAR / MULTI-METRIC PROFILE

### C1 — Team radar, overlaid across expert + all 4 conditions, LOCKED-BUDGET FIGURE

In [ ]:
def compute_radar_metrics(episodes_dict):    widths, lengths, compacts, poss, sprints = [], [], [], [], []    for name, d in episodes_dict.items():        pos = d['left_team'][:, OUTFIELD]        widths.append(pos[:,:,1].std(axis=1)); lengths.append(pos[:,:,0].std(axis=1))        for t in range(0, len(d['steps_left']), 20):            compacts.append(pdist(pos[t]).mean())        poss.append(d['has_possession']); sprints.append(d['sticky_actions'][:,8])    return [np.concatenate(widths).mean(), np.concatenate(lengths).mean(), np.mean(compacts),           np.concatenate(poss).mean()*100, np.concatenate(sprints).mean()*100]metrics_names = ['width','length','compactness','possession%','sprint%']radar_data = {'expert': compute_radar_metrics(expert_episodes)}for cond in CONDITIONS:    radar_data[cond] = compute_radar_metrics(agent_episodes[cond])matrix = np.array(list(radar_data.values()))norm = (matrix - matrix.min(0)) / (matrix.max(0) - matrix.min(0) + 1e-9)angles = np.linspace(0, 2*np.pi, len(metrics_names), endpoint=False).tolist(); angles += angles[:1]fig = plt.figure(figsize=(7,7)); ax = fig.add_subplot(111, polar=True)for label, vals in zip(radar_data.keys(), norm):    v = vals.tolist() + [vals[0]]    ax.plot(angles, v, label=label, color=COND_COLORS.get(label,'black'), linewidth=2)    ax.fill(angles, v, color=COND_COLORS.get(label,'black'), alpha=0.1)ax.set_xticks(angles[:-1]); ax.set_xticklabels(metrics_names)ax.set_title('Team behavioural profile'); ax.legend(loc='upper right', bbox_to_anchor=(1.3,1.1))plt.tight_layout(); plt.show()

### C2 — Radar, late-win vs late-loss overlay per condition, LOCKED-BUDGET FIGURECSI made visually explicit: shape distortion between the two overlaid polygons.

In [ ]:
def compute_radar_metrics_filtered(episodes_dict, score_filter):    widths, lengths, sprints = [], [], []    for name, d in episodes_dict.items():        dscore = d['score_left'] - d['score_right']        elapsed = (1 - d['steps_left']/3001.0) * 90        mask = (elapsed >= 70) & ((dscore > 0) if score_filter=='win' else (dscore < 0))        if mask.sum() == 0:            continue        pos = d['left_team'][mask][:, OUTFIELD]        widths.append(pos[:,:,1].std(axis=1)); lengths.append(pos[:,:,0].std(axis=1))        sprints.append(d['sticky_actions'][mask,8])    if not widths:        return None    return [np.concatenate(widths).mean(), np.concatenate(lengths).mean(), np.concatenate(sprints).mean()*100]metrics_names2 = ['width','length','sprint%']fig, axes = plt.subplots(1, 4, figsize=(20,5), subplot_kw={'projection':'polar'})for ax, cond in zip(axes, CONDITIONS):    win_m = compute_radar_metrics_filtered(agent_episodes[cond], 'win')    loss_m = compute_radar_metrics_filtered(agent_episodes[cond], 'loss')    if win_m is None or loss_m is None:        ax.set_title(f'{cond} -- insufficient late-game data'); continue    matrix = np.array([win_m, loss_m])    norm = (matrix - matrix.min(0)) / (matrix.max(0) - matrix.min(0) + 1e-9)    angles = np.linspace(0, 2*np.pi, 3, endpoint=False).tolist(); angles += angles[:1]    for label, vals, c in zip(['late-win','late-loss'], norm, ['tab:green','tab:red']):        v = vals.tolist() + [vals[0]]        ax.plot(angles, v, label=label, color=c, linewidth=2)        ax.fill(angles, v, color=c, alpha=0.15)    ax.set_xticks(angles[:-1]); ax.set_xticklabels(metrics_names2)    ax.set_title(f'{cond}'); ax.legend(fontsize=8)plt.tight_layout(); plt.show()

---# D. DISTRIBUTIONAL / STATISTICAL — the honest-uncertainty layerAll built from `runs_df` (the 32-row raw per-seed data) -- n=8 per condition,shown explicitly rather than hidden behind an aggregate.

### D1 — Box plot: SAP / CSI / win-rate by condition

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))for ax, metric in zip(axes, ['sap_mean','csi_sap_proxy','win_rate']):    data = [runs_df[runs_df.condition_label==c][metric].dropna() for c in CONDITIONS]    bp = ax.boxplot(data, labels=CONDITIONS, patch_artist=True)    for patch, c in zip(bp['boxes'], CONDITIONS):        patch.set_facecolor(COND_COLORS[c]); patch.set_alpha(0.5)    if metric == 'csi_sap_proxy':        ax.axhline(0, color='gray', lw=0.5)    ax.set_title(metric)plt.tight_layout(); plt.show()

### D2 — Violin plot: full density shape (reveals bimodality a box plot hides)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))for ax, metric in zip(axes, ['sap_mean','csi_sap_proxy','win_rate']):    data = [runs_df[runs_df.condition_label==c][metric].dropna().values for c in CONDITIONS]    ax.violinplot(data, showmeans=True)    ax.set_xticks(range(1,5)); ax.set_xticklabels(CONDITIONS); ax.set_title(metric)plt.tight_layout(); plt.show()

### D3 — Strip/swarm plot: every individual seed shown, mean overlaid (honest about n=8)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))for ax, metric in zip(axes, ['sap_mean','csi_sap_proxy','win_rate']):    for i, c in enumerate(CONDITIONS):        vals = runs_df[runs_df.condition_label==c][metric].dropna()        jitter = np.random.uniform(-0.1, 0.1, len(vals))        ax.scatter(i+jitter, vals, color=COND_COLORS[c], alpha=0.7, s=40)        ax.scatter(i, vals.mean(), color='black', marker='_', s=300, zorder=5)    ax.set_xticks(range(4)); ax.set_xticklabels(CONDITIONS); ax.set_title(f'{metric} (n=8 each)')plt.tight_layout(); plt.show()

### D4 — Forest plot: mean ± 90% CI, LOCKED-BUDGET FIGUREThe single most defensible result figure -- the confirmatory finding, including the null.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))y_pos = np.arange(len(CONDITIONS))means, cis_lo, cis_hi = [], [], []for c in CONDITIONS:    vals = runs_df[runs_df.condition_label==c]['csi_sap_proxy'].dropna().values    m = vals.mean(); se = vals.std(ddof=1)/np.sqrt(len(vals)); ci = 1.645*se    means.append(m); cis_lo.append(m-ci); cis_hi.append(m+ci)ax.errorbar(means, y_pos, xerr=[np.array(means)-np.array(cis_lo), np.array(cis_hi)-np.array(means)],           fmt='o', capsize=5, color='black', markersize=8, zorder=4)for i, c in enumerate(CONDITIONS):    ax.scatter(means[i], i, color=COND_COLORS[c], s=150, zorder=5)ax.axvline(0, color='gray', linestyle='--', label='no context effect')ax.set_yticks(y_pos); ax.set_yticklabels(CONDITIONS)ax.set_xlabel('CSI_SAP (signed, 90% CI)'); ax.set_title('CSI by condition -- mean ± 90% CI')ax.legend(); plt.tight_layout(); plt.show()

---# E. BAR / COMPARATIVE — the headline numbers

### E1 — Grouped bar: SAP by condition x context cell, LOCKED-BUDGET FIGUREThe CSI story in its most literal, plain-English form.

In [ ]:
cells9 = [(tz,sz) for tz in ['early','mid','late'] for sz in ['win','draw','loss']]grid_data = {cond: agent_dfs[cond].groupby(['time_zone','score_zone'])['sprint'].mean()*100 for cond in CONDITIONS}fig, ax = plt.subplots(figsize=(14, 6))x = np.arange(len(cells9)); width = 0.2for i, cond in enumerate(CONDITIONS):    vals = [grid_data[cond].get(cell, np.nan) for cell in cells9]    ax.bar(x + i*width, vals, width, label=cond, color=COND_COLORS[cond])ax.set_xticks(x + 1.5*width); ax.set_xticklabels([f'{tz}\n{sz}' for tz,sz in cells9], fontsize=8)ax.set_ylabel('SAP (%)'); ax.set_title('SAP by condition x context cell'); ax.legend()plt.tight_layout(); plt.show()

### E2 — Gap-closure waterfall: baseline → each condition → human target

In [ ]:
HUMAN_SAP = expert_df['sprint'].mean() * 100BASELINE_SAP = 85.9stages = ['baseline'] + CONDITIONS + ['human target']vals = [BASELINE_SAP] + [cond_summary_dict[c]['sap_mean']['mean'] for c in CONDITIONS] + [HUMAN_SAP]colors_wf = ['gray'] + [COND_COLORS[c] for c in CONDITIONS] + ['black']fig, ax = plt.subplots(figsize=(10, 5))ax.bar(stages, vals, color=colors_wf)for i, v in enumerate(vals):    ax.text(i, v+1, f'{v:.1f}%', ha='center')ax.set_ylabel('SAP (%)'); ax.set_title('Gap closure: baseline -> conditions -> human target')plt.tight_layout(); plt.show()

### E3 — Diverging bar: signed CSI per condition

In [ ]:
csi_means = [cond_summary_dict[c]['csi_sap_proxy']['mean'] for c in CONDITIONS]colors_e3 = ['tab:red' if v < 0 else 'tab:green' for v in csi_means]fig, ax = plt.subplots(figsize=(8, 5))ax.barh(CONDITIONS, csi_means, color=colors_e3)ax.axvline(0, color='black', lw=1)ax.set_xlabel('CSI_SAP (signed)'); ax.set_title('Signed CSI per condition')plt.tight_layout(); plt.show()

### E4 — Stacked bar: win/draw/loss composition per condition (recorded rollouts)

In [ ]:
def episode_outcomes(episodes_dict):    outcomes = {'win': 0, 'draw': 0, 'loss': 0}    for name, d in episodes_dict.items():        final_ds = int(d['score_left'][-1]) - int(d['score_right'][-1])        key = 'win' if final_ds > 0 else ('loss' if final_ds < 0 else 'draw')        outcomes[key] += 1    return outcomeswdl_df = pd.DataFrame({c: episode_outcomes(agent_episodes[c]) for c in CONDITIONS}).Twdl_pct = wdl_df.div(wdl_df.sum(axis=1), axis=0) * 100wdl_pct[['win','draw','loss']].plot(kind='bar', stacked=True, figsize=(9,5),                                    color=['tab:green','tab:gray','tab:red'])plt.ylabel('% of episodes'); plt.title('Win/draw/loss composition (recorded rollouts)')plt.xticks(rotation=0); plt.tight_layout(); plt.show()

### E5 — Sample-size bar: late-win/late-loss step counts behind the CSI estimateMakes the win-rate-driven bin imbalance visible rather than hidden.

In [ ]:
n_lw = [(agent_dfs[c][(agent_dfs[c].time_zone=='late') & (agent_dfs[c].score_zone=='win')]).shape[0] for c in CONDITIONS]n_ll = [(agent_dfs[c][(agent_dfs[c].time_zone=='late') & (agent_dfs[c].score_zone=='loss')]).shape[0] for c in CONDITIONS]fig, ax = plt.subplots(figsize=(9, 5))x = np.arange(len(CONDITIONS)); width = 0.35ax.bar(x-width/2, n_lw, width, label='late-win steps', color='tab:green')ax.bar(x+width/2, n_ll, width, label='late-loss steps', color='tab:red')ax.set_xticks(x); ax.set_xticklabels(CONDITIONS); ax.legend()ax.set_title('Sample size behind the CSI estimate, per condition')plt.tight_layout(); plt.show()

---# F. TEMPORAL / DYNAMIC — behaviour over the match clock

### F1 — SAP by match-minute: condition vs expert vs frozen baseline overlaid

In [ ]:
bins = np.arange(0, 91, 2)fig, ax = plt.subplots(figsize=(12, 5))for cond in CONDITIONS:    sub = agent_dfs[cond]    grp = sub.groupby(pd.cut(sub.elapsed_min, bins), observed=True)['sprint'].mean() * 100    ax.plot(bins[:len(grp)], grp.values, label=cond, color=COND_COLORS[cond])expert_grp = expert_df.groupby(pd.cut(expert_df.elapsed_min, bins), observed=True)['sprint'].mean() * 100ax.plot(bins[:len(expert_grp)], expert_grp.values, label='expert', color='black', linestyle='--', linewidth=2)ax.axhline(85.9, color='gray', linestyle=':', label='frozen baseline')ax.set_xlabel('elapsed minutes'); ax.set_ylabel('SAP (%)')ax.legend(); ax.set_title('SAP by match minute')plt.tight_layout(); plt.show()

### F2 — Score differential + sprint rate, dual axis, one representative episode per condition

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))for ax, cond in zip(axes.flat, CONDITIONS):    name = list(agent_episodes[cond].keys())[0]    d = agent_episodes[cond][name]    elapsed = (1 - d['steps_left']/3001.0) * 90    dscore = d['score_left'] - d['score_right']    sprint = pd.Series(d['sticky_actions'][:,8]).rolling(50, min_periods=1).mean()    ax2 = ax.twinx()    ax.plot(elapsed, dscore, color='black', label='score diff')    ax2.plot(elapsed, sprint, color='tab:orange', alpha=0.7, label='sprint rate')    ax.set_title(f'{cond}'); ax.set_xlabel('elapsed min'); ax.set_ylabel('score diff'); ax2.set_ylabel('sprint rate')plt.tight_layout(); plt.show()

---# G. MOVEMENT / KINEMATIC

### G2 — Movement flow field: average heading per pitch cell, per conditionExtends the notebook cell already built for expert data.

In [ ]:
GRID_X, GRID_Y = 12, 8x_edges = np.linspace(-1, 1, GRID_X+1); y_edges = np.linspace(-0.42, 0.42, GRID_Y+1)x_centers = (x_edges[:-1]+x_edges[1:])/2; y_centers = (y_edges[:-1]+y_edges[1:])/2def flow_field(episodes_dict, players=OUTFIELD):    sum_dx = np.zeros((GRID_Y,GRID_X)); sum_dy = np.zeros((GRID_Y,GRID_X)); count = np.zeros((GRID_Y,GRID_X))    for name, d in episodes_dict.items():        for p in players:            xs, ys = d['left_team'][:,p,0], d['left_team'][:,p,1]            dxs, dys = d['left_team_direction'][:,p,0], d['left_team_direction'][:,p,1]            xi = np.clip(np.digitize(xs,x_edges)-1, 0, GRID_X-1)            yi = np.clip(np.digitize(ys,y_edges)-1, 0, GRID_Y-1)            np.add.at(sum_dx,(yi,xi),dxs); np.add.at(sum_dy,(yi,xi),dys); np.add.at(count,(yi,xi),1)    with np.errstate(invalid='ignore'):        return np.where(count>0,sum_dx/count,0), np.where(count>0,sum_dy/count,0), countfig, axes = plt.subplots(1, 4, figsize=(22, 5))Xc, Yc = np.meshgrid(x_centers, y_centers)for ax, cond in zip(axes, CONDITIONS):    mdx, mdy, cnt = flow_field(agent_episodes[cond])    ax.quiver(Xc, Yc, mdx, mdy, cnt, cmap='viridis', scale=2)    ax.set_xlim(-1,1); ax.set_ylim(-0.42,0.42); ax.invert_yaxis(); ax.set_title(f'{cond}')plt.tight_layout(); plt.show()

---# H. BEHAVIOURAL-STATE TRANSITIONS

### H2 — Transition heatmap: P(sustain sprint) by context cell, per condition

In [ ]:
def sprint_sustain_by_cell(episodes_dict):    cell_names = [(tz,sz) for tz in ['early','mid','late'] for sz in ['win','draw','loss']]    cell_idx = {c: i for i, c in enumerate(cell_names)}    on_on = np.zeros(9); on_total = np.zeros(9)    for name, d in episodes_dict.items():        sprint = d['sticky_actions'][:,8]        elapsed = (1 - d['steps_left']/3001.0) * 90        dscore = d['score_left'] - d['score_right']        tz = np.select([elapsed<20, elapsed<70], ['early','mid'], default='late')        sz = np.select([dscore>0, dscore<0], ['win','loss'], default='draw')        for t in range(1, len(sprint)):            if sprint[t-1] != 1:                continue            idx = cell_idx.get((tz[t], sz[t]))            if idx is not None:                on_total[idx] += 1                on_on[idx] += sprint[t]    return (on_on / np.maximum(on_total, 1)).reshape(3,3), cell_namesfig, axes = plt.subplots(1, 4, figsize=(20, 5))for ax, cond in zip(axes, CONDITIONS):    rate, cell_names = sprint_sustain_by_cell(agent_episodes[cond])    im = ax.imshow(rate, cmap='viridis', vmin=0, vmax=1)    ax.set_xticks(range(3)); ax.set_xticklabels(['win','draw','loss'])    ax.set_yticks(range(3)); ax.set_yticklabels(['early','mid','late'])    ax.set_title(f'{cond}\nP(sustain sprint)')    plt.colorbar(im, ax=ax, shrink=0.7)plt.tight_layout(); plt.show()

---# Not yet built here — extend as needed- **A2** heatmap faceted by all 9 cells (small multiples of A1's grid, looped over `cells9`)- **A5** Voronoi late-win vs late-loss side by side (combine A4's function with C2's masking)- **A11/A12** per-player position ellipse / width-length scatter (per-player version of C1's width/length calc)- **B2/B3** pass network split by context / proximity network (mask `inferred_pass_network`'s loop the same way C2 masks radar)- **C3** percentile "pizza" chart vs human distribution (rank each condition's C1 metrics against expert_df's own per-episode distribution)- **D5** ridge plot of SAP distribution over match-time, layered by condition (`joypy` or manual offset-KDE per condition)- **F3** small-multiples grid, 4 conditions x 9 cells (loop E1's grid into 36 mini histograms)- **G1/G3/G4** speed histogram / sprint-bout-duration histogram / movement rose plot (all derivable from `sticky_actions[:,8]` run-length + `left_team_direction` magnitude/angle)- **H1** Sankey of sprint on/off transitions (feed H2's transition counts into `matplotlib.sankey` or `plotly.graph_objects.Sankey`)- **I1** player-card composite dashboard (subplot grid combining A1 + C1 + key stats per condition, once the above are finalized)